In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd

from src.analytics.ratios import (
    net_profit_margin,
    operating_profit_margin,
    return_on_equity,
    return_on_capital_employed,
    return_on_assets,
)

In [5]:
db_path = PROJECT_ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(db_path)

print("Connected successfully.")

Connected successfully.


In [6]:
sample = pd.read_sql("""
SELECT
    p.company_id,
    p.year,
    p.sales,
    p.operating_profit,
    p.other_income,
    p.net_profit,
    p.interest,

    b.equity_capital,
    b.reserves,
    b.borrowings,
    b.investments,
    b.total_assets

FROM profitandloss p

JOIN balancesheet b
ON p.company_id = b.company_id
AND p.year = b.year

WHERE p.company_id = 'ADANIPORTS'

LIMIT 1;
""", conn)

sample

,company_id,year,sales,operating_profit,other_income,net_profit,interest,equity_capital,reserves,borrowings,investments,total_assets
0,ADANIPORTS,Mar 2013,3577.0,2382.0,344.0,1639.0,542.0,401.0,5993.0,11620.0,222.0,21035.0


In [7]:
row = sample.iloc[0]

print("Company :", row.company_id)
print("Year    :", row.year)

print("\nNet Profit Margin")
print(
    net_profit_margin(
        row.net_profit,
        row.sales
    )
)

print("\nOperating Profit Margin")
print(
    operating_profit_margin(
        row.operating_profit,
        row.sales
    )
)

print("\nROE")
print(
    return_on_equity(
        row.net_profit,
        row.equity_capital,
        row.reserves
    )
)

print("\nROCE")
print(
    return_on_capital_employed(
        row.operating_profit,
        row.interest,
        row.equity_capital,
        row.reserves,
        row.borrowings
    )
)

print("\nROA")
print(
    return_on_assets(
        row.net_profit,
        row.total_assets
    )
)

Company : ADANIPORTS
Year    : Mar 2013

Net Profit Margin
45.820519988817445

Operating Profit Margin
66.59211629857423

ROE
25.633406318423525

ROCE
16.23181969579216

ROA
7.791775612075114


In [8]:
profit = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

balance = pd.read_sql(
    "SELECT * FROM balancesheet",
    conn
)

ratio_df = profit.merge(
    balance,
    on=["company_id", "year"],
    how="inner",
    suffixes=("_pl", "_bs")
)

print(ratio_df.shape)

ratio_df.head()

(1151, 26)


,id_pl,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,61,ABB,Dec 2012,1653.0,1451.0,202.0,12.0,33.0,0.0,19.0,...,21.0,626.0,0.0,260.0,907.0,109.0,1.0,0.0,798.0,907.0
1,62,ABB,Mar 2014,2276.0,2009.0,267.0,12.0,49.0,0.0,22.0,...,21.0,767.0,0.0,351.0,1139.0,98.0,1.0,0.0,1040.0,1139.0
2,63,ABB,Mar 2015,2289.0,1977.0,312.0,14.0,48.0,0.0,15.0,...,21.0,916.0,0.0,436.0,1374.0,96.0,4.0,0.0,1274.0,1374.0
3,64,ABB,Mar 2016,2614.0,2250.0,365.0,14.0,50.0,3.0,14.0,...,21.0,1174.0,0.0,421.0,1616.0,108.0,3.0,0.0,1505.0,1616.0
4,65,ABB,Mar 2017,2903.0,2505.0,398.0,14.0,57.0,2.0,16.0,...,21.0,1366.0,0.0,679.0,2066.0,110.0,6.0,0.0,1950.0,2066.0


In [9]:
ratio_df["net_profit_margin_pct"] = ratio_df.apply(
    lambda row: net_profit_margin(
        row["net_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["operating_profit_margin_pct"] = ratio_df.apply(
    lambda row: operating_profit_margin(
        row["operating_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["return_on_equity_pct"] = ratio_df.apply(
    lambda row: return_on_equity(
        row["net_profit"],
        row["equity_capital"],
        row["reserves"]
    ),
    axis=1
)

ratio_df["return_on_capital_employed_pct"] = ratio_df.apply(
    lambda row: return_on_capital_employed(
        row["operating_profit"],
        row["interest"],
        row["equity_capital"],
        row["reserves"],
        row["borrowings"]
    ),
    axis=1
)

ratio_df["return_on_assets_pct"] = ratio_df.apply(
    lambda row: return_on_assets(
        row["net_profit"],
        row["total_assets"]
    ),
    axis=1
)

print("Profitability ratios calculated successfully.")

Profitability ratios calculated successfully.


In [10]:
ratio_df[
    [
        "company_id",
        "year",
        "net_profit_margin_pct",
        "operating_profit_margin_pct",
        "return_on_equity_pct",
        "return_on_capital_employed_pct",
        "return_on_assets_pct"
    ]
].head(10)

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_capital_employed_pct,return_on_assets_pct
0,ABB,Dec 2012,8.771930,12.220206,22.411128,31.221020,15.986770
1,ABB,Mar 2014,8.699473,11.731107,25.126904,33.883249,17.383670
2,ABB,Mar 2015,10.004369,13.630406,24.439701,33.297759,16.666667
3,ABB,Mar 2016,9.755164,13.963275,21.338912,30.794979,15.779703
4,ABB,Mar 2017,9.541853,13.709955,19.971161,28.839221,13.407551
5,ABB,Mar 2018,12.158884,15.918739,23.685765,31.246308,16.597682
6,ABB,Mar 2019,12.231585,16.444686,22.410359,30.229084,15.300918
7,ABB,Mar 2020,14.488151,18.494991,24.393254,29.393707,16.718354
8,ABB,Mar 2021,16.032483,21.392111,26.556495,34.119782,17.994792
9,ABB,Mar 2022,16.262976,22.023204,28.333333,37.045760,18.915720


In [11]:
ratio_df["opm_difference"] = (
    ratio_df["operating_profit_margin_pct"]
    - ratio_df["opm_percentage"]
).abs()

opm_validation = ratio_df[
    ratio_df["opm_difference"] > 1
].copy()

print(f"Total OPM mismatches (>1%): {len(opm_validation)}")

opm_validation[
    [
        "company_id",
        "year",
        "operating_profit_margin_pct",
        "opm_percentage",
        "opm_difference"
    ]
].head(10)

Total OPM mismatches (>1%): 216


,company_id,year,operating_profit_margin_pct,opm_percentage,opm_difference
22,ADANIENSOL,Mar 2024,34.389113,30.0,4.389113
145,AXISBANK,Mar 2013,30.581614,1353.0,1322.418386
146,AXISBANK,Mar 2014,31.474169,2307.0,2275.525831
147,AXISBANK,Mar 2015,31.362214,3097.0,3065.637786
148,AXISBANK,Mar 2016,32.611984,3466.0,3433.388016
149,AXISBANK,Mar 2017,53.450676,-5715.0,5768.450676
150,AXISBANK,Mar 2018,63.117082,-10277.0,10340.117082
151,AXISBANK,Mar 2019,49.385298,-5447.0,5496.385298
152,AXISBANK,Mar 2020,55.984673,-9859.0,9914.984673
153,AXISBANK,Mar 2021,50.119976,-2510.0,2560.119976


In [12]:
sectors = pd.read_sql(
    """
    SELECT company_id, broad_sector
    FROM sectors
    """,
    conn
)

ratio_df = ratio_df.merge(
    sectors,
    on="company_id",
    how="left"
)

ratio_df[["company_id", "broad_sector"]].head()

,company_id,broad_sector
0,ABB,Industrials
1,ABB,Industrials
2,ABB,Industrials
3,ABB,Industrials
4,ABB,Industrials


In [13]:
opm_validation = ratio_df[
    ratio_df["opm_difference"] > 1
].copy()

In [14]:
opm_validation.groupby("broad_sector").size().sort_values(ascending=False)

broad_sector
Financials                143
Materials                  24
Consumer Discretionary     23
Healthcare                 13
Consumer Staples           12
Energy                      1
dtype: int64

In [15]:
opm_log = opm_validation[
    opm_validation["broad_sector"] != "Financials"
].copy()

print(f"Financial sector mismatches skipped : {len(opm_validation) - len(opm_log)}")
print(f"Mismatches to review                : {len(opm_log)}")

opm_log[
    [
        "company_id",
        "year",
        "broad_sector",
        "operating_profit_margin_pct",
        "opm_percentage",
        "opm_difference"
    ]
].head(10)

Financial sector mismatches skipped : 143
Mismatches to review                : 73


,company_id,year,broad_sector,operating_profit_margin_pct,opm_percentage,opm_difference
22,ADANIENSOL,Mar 2024,Energy,34.389113,30.0,4.389113
299,CIPLA,Mar 2013,Healthcare,73.257640,2214.0,2140.742360
300,CIPLA,Mar 2014,Healthcare,78.895115,2147.0,2068.104885
301,CIPLA,Mar 2015,Healthcare,80.943147,2163.0,2082.056853
302,CIPLA,Mar 2016,Healthcare,82.015954,2480.0,2397.984046
303,CIPLA,Mar 2017,Healthcare,82.659441,2496.0,2413.340559
304,CIPLA,Mar 2018,Healthcare,81.347321,2826.0,2744.652679
305,CIPLA,Mar 2019,Healthcare,81.071996,3097.0,3015.928004
306,CIPLA,Mar 2020,Healthcare,81.286481,3206.0,3124.713519
307,CIPLA,Mar 2021,Healthcare,77.802714,4252.0,4174.197286


In [16]:
opm_log.to_csv(
    PROJECT_ROOT / "output" / "opm_validation_log.csv",
    index=False
)

print("OPM validation log saved.")

OPM validation log saved.


In [17]:
ratio_df["broad_sector"].value_counts()

broad_sector
Financials                294
Energy                    170
Consumer Discretionary    165
Industrials               129
Materials                 118
Consumer Staples           84
Healthcare                 72
Information Technology     71
Communication Services     24
Real Estate                24
Name: count, dtype: int64

In [18]:
ratio_df["roce_benchmark"] = ratio_df["broad_sector"].apply(
    lambda sector: "Sector Relative"
    if sector == "Financials"
    else "Absolute"
)

ratio_df[
    ["company_id",
     "broad_sector",
     "return_on_capital_employed_pct",
     "roce_benchmark"]
].head(15)

,company_id,broad_sector,return_on_capital_employed_pct,roce_benchmark
0,ABB,Industrials,31.221020,Absolute
1,ABB,Industrials,33.883249,Absolute
2,ABB,Industrials,33.297759,Absolute
3,ABB,Industrials,30.794979,Absolute
4,ABB,Industrials,28.839221,Absolute
5,ABB,Industrials,31.246308,Absolute
6,ABB,Industrials,30.229084,Absolute
7,ABB,Industrials,29.393707,Absolute
8,ABB,Industrials,34.119782,Absolute
9,ABB,Industrials,37.045760,Absolute


In [19]:
import importlib
import src.analytics.ratios as ratios

importlib.reload(ratios)

<module 'src.analytics.ratios' from 'c:\\Users\\panka\\OneDrive\\Desktop\\Nifty100_Project\\src\\analytics\\ratios.py'>

In [20]:
row = sample.iloc[0]

In [21]:
print("Company :", row.company_id)
print("Year    :", row.year)

print("\nDebt to Equity")
print(
    ratios.debt_to_equity(
        row.borrowings,
        row.equity_capital,
        row.reserves
    )
)

print("\nInterest Coverage Ratio")
print(
    ratios.interest_coverage_ratio(
        row.operating_profit,
        row.other_income,
        row.interest
    )
)

print("\nNet Debt")
print(
    ratios.net_debt(
        row.borrowings,
        row.investments
    )
)

print("\nAsset Turnover")
print(
    ratios.asset_turnover(
        row.sales,
        row.total_assets
    )
)

Company : ADANIPORTS
Year    : Mar 2013

Debt to Equity
1.817328745699093

Interest Coverage Ratio
5.029520295202952

Net Debt
11398.0

Asset Turnover
0.17004991680532447


In [22]:
ratio_df["debt_to_equity"] = ratio_df.apply(
    lambda row: ratios.debt_to_equity(
        row.borrowings,
        row.equity_capital,
        row.reserves
    ),
    axis=1
)

ratio_df["interest_coverage"] = ratio_df.apply(
    lambda row: ratios.interest_coverage_ratio(
        row.operating_profit,
        row.other_income,
        row.interest
    ),
    axis=1
)

ratio_df["net_debt"] = ratio_df.apply(
    lambda row: ratios.net_debt(
        row.borrowings,
        row.investments
    ),
    axis=1
)

ratio_df["asset_turnover"] = ratio_df.apply(
    lambda row: ratios.asset_turnover(
        row.sales,
        row.total_assets
    ),
    axis=1
)

print("Leverage & Efficiency KPIs calculated successfully.")

Leverage & Efficiency KPIs calculated successfully.


In [23]:
ratio_df[
    [
        "company_id",
        "year",
        "debt_to_equity",
        "interest_coverage",
        "net_debt",
        "asset_turnover"
    ]
].head(10)

,company_id,year,debt_to_equity,interest_coverage,net_debt,asset_turnover
0,ABB,Dec 2012,0.000000,NaN,0.0,1.822492
1,ABB,Mar 2014,0.000000,NaN,0.0,1.998244
2,ABB,Mar 2015,0.000000,NaN,0.0,1.665939
3,ABB,Mar 2016,0.000000,138.333333,0.0,1.617574
4,ABB,Mar 2017,0.000000,227.500000,0.0,1.405131
5,ABB,Mar 2018,0.000000,160.500000,0.0,1.365066
6,ABB,Mar 2019,0.000000,359.000000,0.0,1.250935
7,ABB,Mar 2020,0.071987,96.777778,175.0,1.153933
8,ABB,Mar 2021,0.058801,55.722222,153.0,1.122396
9,ABB,Mar 2022,0.053901,61.315789,152.0,1.163116


In [24]:
ratio_df["high_leverage_flag"] = (
    (ratio_df["debt_to_equity"] > 5) &
    (ratio_df["broad_sector"] != "Financials")
)

ratio_df["icr_label"] = ratio_df["interest_coverage"].apply(
    lambda x: "Debt Free" if pd.isna(x) else None
)

ratio_df["icr_warning"] = ratio_df["interest_coverage"].apply(
    lambda x: False if pd.isna(x) else x < 1.5
)

print("Leverage flags created.")

Leverage flags created.


In [25]:
import importlib
import src.analytics.cagr as cagr

importlib.reload(cagr)

<module 'src.analytics.cagr' from 'c:\\Users\\panka\\OneDrive\\Desktop\\Nifty100_Project\\src\\analytics\\cagr.py'>

In [26]:
print(cagr.calculate_cagr(100, 200, 5))
print(cagr.calculate_cagr(100, -20, 5))
print(cagr.calculate_cagr(-50, 120, 5))
print(cagr.calculate_cagr(0, 120, 5))

(14.869835499703509, 'NORMAL')
(None, 'DECLINE_TO_LOSS')
(None, 'TURNAROUND')
(None, 'ZERO_BASE')


In [27]:
profit_history = pd.read_sql("""
SELECT
    company_id,
    year,
    sales,
    net_profit,
    eps
FROM profitandloss
ORDER BY company_id, year
""", conn)

profit_history.head(20)

,company_id,year,sales,net_profit,eps
0,ABB,Dec 2012,1653.0,145.0,68.0
1,ABB,Mar 2014,2276.0,198.0,93.0
2,ABB,Mar 2015,2289.0,229.0,108.0
3,ABB,Mar 2016,2614.0,255.0,120.0
4,ABB,Mar 2017,2903.0,277.0,130.0
5,ABB,Mar 2018,3298.0,401.0,189.0
6,ABB,Mar 2019,3679.0,450.0,212.0
7,ABB,Mar 2020,4093.0,593.0,279.0
8,ABB,Mar 2021,4310.0,691.0,325.0
9,ABB,Mar 2022,4913.0,799.0,376.0


In [28]:
profit_history.groupby("company_id").size().describe()

count    92.000000
mean     12.793478
std       1.969761
min       3.000000
25%      13.000000
50%      13.000000
75%      13.000000
max      26.000000
dtype: float64

In [32]:
profit_history[
    profit_history["year"].str.extract(r'(\d{4})')[0].isna()
][["company_id", "year"]]


,company_id,year
12,ABB,TTM
24,ADANIENSOL,TTM
37,ADANIENT,TTM
46,ADANIGREEN,TTM
71,ADANIPORTS,TTM
...,...,...
1124,TECHM,TTM
1137,TITAN,TTM
1150,TORNTPHARM,TTM
1163,TRENT,TTM


In [33]:
profit_history = profit_history[
    profit_history["year"] != "TTM"
].copy()

In [34]:
profit_history["year_num"] = (
    profit_history["year"]
    .str.extract(r'(\d{4})')[0]
    .astype(int)
)

profit_history = (
    profit_history
    .sort_values(["company_id", "year_num"])
    .reset_index(drop=True)
)

profit_history[
    ["company_id", "year", "year_num"]
].head(20)

,company_id,year,year_num
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
2,ABB,Mar 2015,2015
3,ABB,Mar 2016,2016
4,ABB,Mar 2017,2017
5,ABB,Mar 2018,2018
6,ABB,Mar 2019,2019
7,ABB,Mar 2020,2020
8,ABB,Mar 2021,2021
9,ABB,Mar 2022,2022


In [35]:
revenue_cagr_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_5yr"] = None
    group["revenue_cagr_5yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 5]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            5
        )

        group.loc[i, "revenue_cagr_5yr"] = cagr_value
        group.loc[i, "revenue_cagr_5yr_flag"] = flag

    revenue_cagr_results.append(group)

revenue_cagr_df = pd.concat(
    revenue_cagr_results,
    ignore_index=True
)

print(revenue_cagr_df.shape)

revenue_cagr_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_5yr",
        "revenue_cagr_5yr_flag"
    ]
].head(20)

(1085, 8)


,company_id,year,sales,revenue_cagr_5yr,revenue_cagr_5yr_flag
0,ABB,Dec 2012,1653.0,None,INSUFFICIENT
1,ABB,Mar 2014,2276.0,None,INSUFFICIENT
2,ABB,Mar 2015,2289.0,None,INSUFFICIENT
3,ABB,Mar 2016,2614.0,None,INSUFFICIENT
4,ABB,Mar 2017,2903.0,11.921839,NORMAL
5,ABB,Mar 2018,3298.0,None,INSUFFICIENT
6,ABB,Mar 2019,3679.0,10.080782,NORMAL
7,ABB,Mar 2020,4093.0,12.325715,NORMAL
8,ABB,Mar 2021,4310.0,10.518336,NORMAL
9,ABB,Mar 2022,4913.0,11.09639,NORMAL


In [ ]:
revenue_cagr_3_results = []

for company, group in profit_history.groupby("company_id"):

    group = group.sort_values("year_num").reset_index(drop=True)

    group["revenue_cagr_3yr"] = None
    group["revenue_cagr_3yr_flag"] = "INSUFFICIENT"

    for i in range(len(group)):

        current_year = group.loc[i, "year_num"]

        previous = group[group["year_num"] == current_year - 3]

        if previous.empty:
            continue

        start_sales = previous.iloc[0]["sales"]
        end_sales = group.loc[i, "sales"]

        cagr_value, flag = cagr.calculate_cagr(
            start_sales,
            end_sales,
            3
        )

        group.loc[i, "revenue_cagr_3yr"] = cagr_value
        group.loc[i, "revenue_cagr_3yr_flag"] = flag

    revenue_cagr_3_results.append(group)

revenue_cagr_3_df = pd.concat(
    revenue_cagr_3_results,
    ignore_index=True
)

print(revenue_cagr_3_df.shape)

revenue_cagr_3_df[
    [
        "company_id",
        "year",
        "sales",
        "revenue_cagr_3yr",
        "revenue_cagr_3yr_flag"
    ]
].head(20)

NameError: name 'revenue_cagr_3_results' is not defined